# 🔬 Test Stable Diffusion x4 Upscaler

**Model:** [`stabilityai/stable-diffusion-x4-upscaler`](https://huggingface.co/stabilityai/stable-diffusion-x4-upscaler)  
**Pipeline:** `StableDiffusionUpscalePipeline` (diffusers)  
**Upscale:** x4 (text-guided, latent diffusion)  
**Necesită:** CUDA GPU

---

## 1. Setup

In [ ]:
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFilter
import matplotlib.pyplot as plt
!pip install diffusers transformers scikit-image --quiet
from diffusers import StableDiffusionUpscalePipeline
from skimage.metrics import peak_signal_noise_ratio, structural_similarity, mean_squared_error
import time, os, psutil, io

assert torch.cuda.is_available(), "CUDA necesar! Rulează 01_install_dependencies.ipynb cu varianta CUDA."
device = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\redis\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce GTX 1650


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

## 2. Încărcare Model

In [ ]:
MODEL_ID = "stabilityai/stable-diffusion-x4-upscaler"
mem_before = psutil.Process().memory_info().rss / (1024**2)

# Load pipeline in float32 to avoid NaN/black output from VAE
pipe = StableDiffusionUpscalePipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()  # reduce VRAM usage

# ✅ FIX 1: Disable safety checker — it blanks outputs to black for
#    synthetic/simple images (gradients, shapes) by false-positive triggering.
pipe.safety_checker = None

# ✅ FIX 2: Also disable the watermarker which can corrupt pixel values
if hasattr(pipe, "watermark"):
    pipe.watermark = None

mem_after = psutil.Process().memory_info().rss / (1024**2)
print(f"✅ Model incarcat! RAM delta: {mem_after - mem_before:.0f} MB")
print(f"VRAM alocat: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")
print("⚠️  Safety checker dezactivat (necesar pentru imagini sintetice/mock)")

## 3. Mock Data
Imagini LR 128×128. Modelul le face x4 → 512×512.  
**🔄 Înlocuiește cu propriile tale imagini.**

In [ ]:
def make_gradient(w, h):
    img = np.zeros((h, w, 3), dtype=np.uint8)
    for x in range(w):
        r = x / (w-1)
        img[:, x] = [int(255*(1-r)), int(50+150*r), int(255*r)]
    return Image.fromarray(img)

def make_shapes(w, h):
    img = Image.new("RGB", (w, h), (30, 30, 60))
    d = ImageDraw.Draw(img)
    d.ellipse([8, 8, w//2-4, h//2-4], fill=(220, 60, 60))
    d.rectangle([w//2+4, 8, w-8, h//2-4], fill=(60, 180, 75))
    d.polygon([(w//4, h-8), (w//2, h//2+8), (3*w//4, h-8)], fill=(65, 105, 225))
    return img

def make_checker(w, h, sq=16):
    img = np.zeros((h, w, 3), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            img[y, x] = [200]*3 if ((x//sq)+(y//sq))%2==0 else [50]*3
    return Image.fromarray(img)

LR_SIZE = 128
SCALE = 4
HR_SIZE = LR_SIZE * SCALE

# ============================================================
# 🔄 MOCK DATA — Înlocuiește:
#    mock_images["Nume"] = (Image.open("cale.jpg").resize((128,128)), "prompt text")
# ============================================================
mock_images = {
    "Gradient": (make_gradient(LR_SIZE, LR_SIZE), "a smooth colorful gradient, high quality"),
    "Forme Geometrice": (make_shapes(LR_SIZE, LR_SIZE), "geometric shapes, circles and rectangles, clean digital art"),
    "Checkerboard": (make_checker(LR_SIZE, LR_SIZE), "a black and white checkerboard pattern, sharp edges"),
}

fig, axes = plt.subplots(1, len(mock_images), figsize=(4*len(mock_images), 4))
for ax, (name, (img, _)) in zip(axes, mock_images.items()):
    ax.imshow(img); ax.set_title(f"{name}\n{img.size[0]}x{img.size[1]}"); ax.axis("off")
plt.suptitle("Mock LR Images (Input)", fontweight="bold"); plt.tight_layout(); plt.show()

## 4. Funcții Metrici

In [ ]:
def calc_psnr(a, b): return peak_signal_noise_ratio(np.array(a).astype(np.float64), np.array(b).astype(np.float64), data_range=255)
def calc_ssim(a, b): return structural_similarity(np.array(a).astype(np.float64), np.array(b).astype(np.float64), data_range=255, channel_axis=-1)
def calc_mse(a, b): return mean_squared_error(np.array(a).astype(np.float64), np.array(b).astype(np.float64))
def calc_rmse(a, b): return np.sqrt(calc_mse(a, b))
def calc_mae(a, b): return np.mean(np.abs(np.array(a).astype(np.float64) - np.array(b).astype(np.float64)))

def calc_sharpness(img):
    g = img.convert("L")
    lap = g.filter(ImageFilter.Kernel((3,3),[0,1,0,1,-4,1,0,1,0],1,128))
    return np.var(np.array(lap).astype(np.float64) - 128.0)

def calc_entropy(img):
    h, _ = np.histogram(np.array(img.convert("L")).flatten(), bins=256, range=(0,255), density=True)
    h = h[h > 0]; return -np.sum(h * np.log2(h))

def calc_contrast(img): return np.std(np.array(img.convert("L")).astype(np.float64))

def calc_edge_density(img):
    e = np.array(img.convert("L").filter(ImageFilter.FIND_EDGES))
    return np.sum(e > 30) / e.size * 100

def calc_delta_e(a, b):
    from skimage.color import rgb2lab
    la = rgb2lab(np.array(a).astype(np.float64)/255.0)
    lb = rgb2lab(np.array(b).astype(np.float64)/255.0)
    d = np.sqrt(np.sum((la-lb)**2, axis=-1))
    return {"mean": np.mean(d), "median": np.median(d), "max": np.max(d)}

def file_size(img, fmt="PNG"):
    buf = io.BytesIO(); img.save(buf, format=fmt); return buf.tell()

print("✅ Metrici definite.")

## 5. Rulare Upscale + Metrici

In [ ]:
NUM_INFERENCE_STEPS = 25  # mai puțin = mai rapid, mai mult = calitate mai bună
GUIDANCE_SCALE = 7.5

# ✅ FIX 3: Helper to validate output — detects all-black / NaN outputs
def validate_sr_output(sr_img, name):
    arr = np.array(sr_img)
    mean_val = arr.mean()
    if mean_val < 1.0:
        print(f"   ⚠️  WARNING: SR output is nearly black (mean={mean_val:.4f}) for '{name}'!")
        print(f"      Possible causes: safety checker, VAE NaN, or VRAM overflow.")
    elif np.isnan(arr.astype(np.float32)).any():
        print(f"   ⚠️  WARNING: SR output contains NaN values for '{name}'!")
    else:
        print(f"   ✅ SR output OK (mean pixel={mean_val:.1f}, min={arr.min()}, max={arr.max()})")
    return arr

results = {}
for name, (lr_img, prompt) in mock_images.items():
    print(f"\n{'='*60}")
    print(f"🔄 {name} | Prompt: '{prompt}'")
    
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()

    # ✅ FIX 4: generator for reproducibility (helps with NaN debugging)
    generator = torch.Generator(device=device).manual_seed(42)

    raw_output = pipe(
        prompt=prompt,
        image=lr_img,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    )
    sr_img = raw_output.images[0]
    t_total = time.perf_counter() - t0
    gpu_peak = torch.cuda.max_memory_allocated() / (1024**2)

    # ✅ FIX 5: Clamp pixel values to [0, 255] and force uint8
    #    (VAE can produce slight out-of-range values that PIL clips to 0)
    sr_arr = np.clip(np.array(sr_img), 0, 255).astype(np.uint8)
    sr_img = Image.fromarray(sr_arr)

    # Validate — print warning if still black
    validate_sr_output(sr_img, name)
    
    # Bicubic baseline
    bicubic_img = lr_img.resize(sr_img.size, Image.BICUBIC)
    
    # Metrici no-reference
    nr = {}
    for label, img in [("LR", lr_img), ("Bicubic", bicubic_img), ("SD x4", sr_img)]:
        nr[label] = {"Sharpness": calc_sharpness(img), "Entropy": calc_entropy(img),
                     "Contrast": calc_contrast(img), "Edge %": calc_edge_density(img)}
    
    results[name] = {
        "lr": lr_img, "sr": sr_img, "bicubic": bicubic_img, "prompt": prompt,
        "time_s": t_total, "gpu_peak_mb": gpu_peak,
        "throughput_px_s": (sr_img.size[0]*sr_img.size[1]) / t_total,
        "nr_metrics": nr,
        "file_sizes": {"LR": file_size(lr_img), "SR": file_size(sr_img), "Bicubic": file_size(bicubic_img)},
    }
    
    print(f"   {lr_img.size[0]}x{lr_img.size[1]} → {sr_img.size[0]}x{sr_img.size[1]}")
    print(f"   Timp: {t_total:.2f}s | GPU peak: {gpu_peak:.0f} MB")

print(f"\n{'='*60}\n✅ Toate imaginile procesate!")

## 6. Comparație Vizuală: LR → Bicubic → SD x4

In [ ]:
n = len(results)
fig, axes = plt.subplots(3, n, figsize=(5*n, 14))
for idx, (name, d) in enumerate(results.items()):
    axes[0,idx].imshow(d["lr"]); axes[0,idx].set_title(f"{name}\nLR {d['lr'].size[0]}x{d['lr'].size[1]}", fontsize=9); axes[0,idx].axis("off")
    axes[1,idx].imshow(d["bicubic"]); axes[1,idx].set_title(f"Bicubic\n{d['bicubic'].size[0]}x{d['bicubic'].size[1]}", fontsize=9); axes[1,idx].axis("off")
    axes[2,idx].imshow(d["sr"]); axes[2,idx].set_title(f"SD x4 Upscaler\n{d['sr'].size[0]}x{d['sr'].size[1]}\n{d['time_s']:.2f}s", fontsize=9); axes[2,idx].axis("off")
for ax, l in zip(axes[:,0], ["Input (LR)", "Bicubic", "SD x4 (AI)"]):
    ax.set_ylabel(l, fontsize=11, fontweight="bold", rotation=90, labelpad=15)
plt.suptitle("LR → Bicubic → Stable Diffusion x4 Upscaler", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

## 7. Metrici No-Reference

In [ ]:
print("="*90)
print("METRICI NO-REFERENCE (Sharpness, Entropy, Contrast, Edge Density)")
print("="*90)
for metric in ["Sharpness", "Entropy", "Contrast", "Edge %"]:
    print(f"\n  {metric}")
    print(f"  {'Imagine':<25} {'LR':>10} {'Bicubic':>10} {'SD x4':>10}")
    print(f"  {'─'*60}")
    for name, d in results.items():
        nr = d["nr_metrics"]
        print(f"  {name:<25} {nr['LR'][metric]:>10.3f} {nr['Bicubic'][metric]:>10.3f} {nr['SD x4'][metric]:>10.3f}")

## 8. Grafice No-Reference

In [ ]:
names = list(results.keys())
metrics_list = ["Sharpness", "Entropy", "Contrast", "Edge %"]
x = np.arange(len(names)); w = 0.25

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, metric in zip(axes.flat, metrics_list):
    lr_v = [results[n]["nr_metrics"]["LR"][metric] for n in names]
    bic_v = [results[n]["nr_metrics"]["Bicubic"][metric] for n in names]
    sr_v = [results[n]["nr_metrics"]["SD x4"][metric] for n in names]
    ax.bar(x-w, lr_v, w, label="LR", color="#9E9E9E")
    ax.bar(x, bic_v, w, label="Bicubic", color="#FF9800")
    ax.bar(x+w, sr_v, w, label="SD x4", color="#4CAF50")
    ax.set_title(metric, fontweight="bold"); ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha="right", fontsize=8)
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
plt.suptitle("No-Reference Metrics: LR vs Bicubic vs SD x4", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 9. Histograme Per Canal

In [ ]:
ch_names = ["Red", "Green", "Blue"]
ch_colors = ["#EF5350", "#66BB6A", "#42A5F5"]
for img_name, d in results.items():
    fig, axes = plt.subplots(3, 3, figsize=(14, 8))
    fig.suptitle(f"Histograme — {img_name}", fontsize=12, fontweight="bold")
    for col, (label, img) in enumerate([("LR", d["lr"]), ("Bicubic", d["bicubic"]), ("SD x4", d["sr"])]):
        arr = np.array(img)
        for ch in range(3):
            axes[ch, col].hist(arr[:,:,ch].flatten(), bins=64, range=(0,255), color=ch_colors[ch], alpha=0.7)
            axes[ch, col].set_xlim(0, 255)
            if ch == 0: axes[ch, col].set_title(label, fontweight="bold")
            if col == 0: axes[ch, col].set_ylabel(ch_names[ch], fontweight="bold")
    plt.tight_layout(); plt.show()

## 10. Performanță

In [ ]:
print("="*90)
print("PERFORMANȚĂ")
print("="*90)
print(f"{'Imagine':<25} {'LR Size':<12} {'SR Size':<12} {'Timp (s)':<10} {'GPU Peak':>10} {'Throughput':>14}")
print(f"{'─'*85}")
times = []
for name, d in results.items():
    lr = d['lr'].size; sr = d['sr'].size; t = d['time_s']; times.append(t)
    print(f"{name:<25} {lr[0]}x{lr[1]:<8} {sr[0]}x{sr[1]:<8} {t:<10.2f} {d['gpu_peak_mb']:>8.0f} MB {d['throughput_px_s']/1e3:>11.1f} Kpx/s")
print(f"{'─'*85}")
print(f"{'MEDIE':<25} {'':12} {'':12} {np.mean(times):<10.2f}")
print(f"\nModel: {MODEL_ID}")
print(f"Steps: {NUM_INFERENCE_STEPS} | Guidance: {GUIDANCE_SCALE}")

## 11. File Sizes

In [ ]:
print(f"{'Imagine':<25} {'LR (PNG)':>12} {'Bicubic':>12} {'SD x4':>12} {'Ratio SR/LR':>12}")
print(f"{'─'*75}")
for name, d in results.items():
    fs = d['file_sizes']
    print(f"{name:<25} {fs['LR']:>10} B {fs['Bicubic']:>10} B {fs['SR']:>10} B {fs['SR']/fs['LR']:>11.2f}x")

## 12. Sumar

In [ ]:
print("█"*80)
print("  SUMAR — Stable Diffusion x4 Upscaler")
print("█"*80)
print(f"  Model:          {MODEL_ID}")
print(f"  Upscale:        x4 (text-guided latent diffusion)")
print(f"  Steps:          {NUM_INFERENCE_STEPS}")
print(f"  Guidance scale: {GUIDANCE_SCALE}")
print(f"  Device:         {torch.cuda.get_device_name(0)}")
print(f"  Timp mediu:     {np.mean(times):.2f}s")
avg_sharp_lr = np.mean([results[n]['nr_metrics']['LR']['Sharpness'] for n in results])
avg_sharp_sr = np.mean([results[n]['nr_metrics']['SD x4']['Sharpness'] for n in results])
avg_sharp_bic = np.mean([results[n]['nr_metrics']['Bicubic']['Sharpness'] for n in results])
print(f"\n  {'Metrică':<20} {'LR':>10} {'Bicubic':>10} {'SD x4':>10}")
print(f"  {'─'*55}")
for m in ["Sharpness", "Entropy", "Contrast", "Edge %"]:
    lr_v = np.mean([results[n]['nr_metrics']['LR'][m] for n in results])
    bic_v = np.mean([results[n]['nr_metrics']['Bicubic'][m] for n in results])
    sr_v = np.mean([results[n]['nr_metrics']['SD x4'][m] for n in results])
    print(f"  {m:<20} {lr_v:>10.3f} {bic_v:>10.3f} {sr_v:>10.3f}")
print(f"\n  Notă: Acest model e text-guided (generativ), deci adaugă detalii")
print(f"  bazate pe prompt. Nu este un SR pur — nu păstrează 1:1 conținutul.")
print("█"*80)

## 13. Salvare

In [ ]:
OUT = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output_sd_x4")
os.makedirs(OUT, exist_ok=True)
for name, d in results.items():
    s = name.replace(" ","_")
    d["lr"].save(os.path.join(OUT, f"{s}_LR.png"))
    d["sr"].save(os.path.join(OUT, f"{s}_SR.png"))
    d["bicubic"].save(os.path.join(OUT, f"{s}_Bicubic.png"))
    print(f"Salvat: {s}")
print(f"\n✅ Rezultate în: {OUT}")

---
## 📝 Note
- **Text-guided:** Modelul folosește un prompt text pt a ghida upscale-ul (adaugă detalii generative)
- **Diferit de Swin2SR:** Acesta e generativ (latent diffusion), nu doar reconstructiv
- **VRAM:** ~5-7 GB minim. Folosește `enable_attention_slicing()` pt mai puțin VRAM
- **Steps:** Mai multe steps = calitate mai bună, dar mai lent
- **Mock data:** Înlocuiește `mock_images` din secțiunea 3